# Assignment 1

Deadline: 19.03.2026, 12:00 CET

Dan Friedman, student-id, dannevo.friedman@uzh.ch

In [1]:
# Import standard libraries
import os
import sys
import time
from typing import Optional
from pathlib import Path

# Import third-party libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as pltx

# Import local modules

current_dir = Path(os.getcwd())
project_root = current_dir.parent
src_path = project_root / 'src'

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from estimation.covariance import Covariance, is_pos_def, make_pos_def
from estimation.expected_return import ExpectedReturn
from optimization.constraints import Constraints
from optimization.optimization import Optimization, Objective
from optimization.optimization_data import OptimizationData
from optimization.quadratic_program import QuadraticProgram, USABLE_SOLVERS
from helper_functions import simulate_correlated_gbm

ModuleNotFoundError: No module named 'numpy'

## 1. Solver Horse Race

### 1.a)
(3 points)

Generate a synthetic dataset of dimension TxN, T=1000, N=50, and compute a vector of expected returns, q, and a covariance matrix, P, using classes ExpectedReturn and Covariance respectively.

In [14]:

# Set the dimensions
T = 1000       # Number of time steps
N = 50         # Number of assets
rnd_seed = 42  # Random seed for reproducibility

# Set random seed for reproducibility
np.random.seed(rnd_seed)

# Generate a random mean vector and covariance matrix.
# Make sure the covariance matrix is positive definite.

mu = np.random.rand(N) * 0.05  # Random expected returns
sigma = np.random.rand(N, N) * 0.01  # Random covariance matrix
sigma = sigma @ sigma.T  # Make it symmetric
sigma = make_pos_def(sigma)  # Make sure it's positive definite

# Generate correlated geometric Brownian motion paths and compute discrete returns
prices = simulate_correlated_gbm(mu=mu, sigma=sigma, T=T, random_seed=None)
returns = prices.pct_change().dropna()

# Compute the vector of expected returns from df using class ExpectedReturn
q = ExpectedReturn(method='geometric').estimate(returns, inplace=False)

# Compute the covariance matrix from df using class Covariance
P = Covariance(check_positive_definite=True).estimate(sigma, inplace=False)

# Display the results
print("Vector of expected returns (q):")
print(q)

print("\nCovariance matrix (P):")
print(P.shape, P)

Vector of expected returns (q):
Asset_1     6.834584e-07
Asset_2     4.813004e-05
Asset_3    -2.629382e-06
Asset_4     7.194294e-06
Asset_5    -1.417871e-05
Asset_6    -1.331390e-05
Asset_7    -8.219307e-06
Asset_8     2.202089e-05
Asset_9    -1.314271e-06
Asset_10    1.932581e-05
Asset_11   -3.832428e-05
Asset_12   -2.004402e-05
Asset_13    3.441596e-05
Asset_14   -1.967441e-05
Asset_15    1.967562e-05
Asset_16   -4.279296e-05
Asset_17   -1.865479e-05
Asset_18   -9.757285e-06
Asset_19    3.022822e-05
Asset_20   -2.366421e-06
Asset_21    1.244481e-05
Asset_22   -2.643783e-05
Asset_23   -2.047332e-05
Asset_24    1.107929e-05
Asset_25    3.173375e-05
Asset_26    1.544018e-05
Asset_27   -2.518845e-08
Asset_28   -2.789889e-05
Asset_29    2.786196e-05
Asset_30   -2.008140e-05
Asset_31    6.338346e-06
Asset_32   -2.434529e-05
Asset_33   -4.609004e-05
Asset_34    2.864719e-05
Asset_35    6.216258e-05
Asset_36    8.226286e-06
Asset_37   -3.289507e-05
Asset_38   -7.112443e-06
Asset_39    8.0855

### 1.b)
(3 points)

Instantiate a constraints object by injecting column names of the return series created in 1.a) as ids and add:
- a budget constaint (i.e., asset weights have to sum to one)
- lower bounds of 0.0 for all assets
- upper bounds of 0.2 for all assets
- group contraints such that the sum of the weights of the first 15 assets is <= 0.3, the sum of assets 16 to 45 is <= 0.4 and the sum of assets 41 to 50 is <= 0.5

In [15]:
# Instantiate the Constraints class
constraints = Constraints(ids = returns.columns.tolist())
constraints.selection = constraints.ids

# Add budget constraint
constraints.add_budget()

# Add box constraints (i.e., lower and upper bounds)
constraints.add_box(lower=0, upper=0.2)

# Add linear constraints

ids = returns.columns

# group 1: sum(w) of assets 1-15 <= 0.3
g1 = pd.Series(0, index=ids)
g1.iloc[0:15] = 1
# Group 2: sum(w) of assets 16–39 ≤ 0.4
g2 = pd.Series(0, index=ids)
g2.iloc[15:45] = 1
# Group 3: sum(w) of assets 41–50 ≤ 0.5
g3 = pd.Series(0, index=ids)
g3.iloc[41:50] = 1

constraints.add_linear(g_values=g1, sense='<=', rhs=0.3)
constraints.add_linear(g_values=g2, sense='<=', rhs=0.4)
constraints.add_linear(g_values=g3, sense='<=', rhs=0.5)

# Display some columns of the G matrix to verify that the constraints have been set up correctly
constraints.linear['G'][['Asset_1', 'Asset_15', 'Asset_16', 'Asset_40', 'Asset_41', 'Asset_50']]

,Asset_1,Asset_15,Asset_16,Asset_40,Asset_41,Asset_50
0,1,1,0,0,0,0
0,0,0,1,1,1,0
0,0,0,0,0,0,1


### 1.c) 
(4 points)

Solve a Mean-Variance optimization problem (using coefficients P and q in the objective function) which satisfies the above defined constraints.
Repeat the task for all open-source solvers in qpsolvers that you could install and compare the results in terms of:

- runtime
- accuracy: value of the primal problem.
- reliability: are all constraints fulfilled? Extract primal residuals, dual residuals and duality gap.

Generate a DataFrame with the solvers as column names and the following row index: 'solution_found': bool, 'objective': float, 'primal_residual': float, 'dual_residual': float, 'duality_gap': float, 'runtime': float.

Put NA's for solvers where the optimization failed for some reason.




In [17]:
import qpsolvers
from qpsolvers import solve_qp
print("avaliable solvers:", qpsolvers.available_solvers)
# Extract the constraints in the format required by the solver
GhAb = constraints.to_GhAb(lbub_to_G=True)

# Define a dictionary to store the results in case a solver fails
result_on_fail =  {
    'solution_found': False,
    'objective_builtin': np.nan,
    'primal_residual': np.nan,
    'dual_residual' : np.nan,
    'duality_gap': np.nan,
    'runtime': np.nan,
}
P_qp = np.asarray(P)
#P_qp += 1e-8 * np.eye(P_qp.shape[0])
q_qp = np.asarray(q).reshape(-1) #flipped for objective formulation
print("P shape:", P_qp.shape)
print("q shape:", q_qp.shape)

def objective(P_qp, q_qp, w):
    return 0.5 * w @ P_qp @ w - q_qp @ w

# Constraints (converted to numpy arrays if they exist, otherwise set to None)
G = np.asarray(GhAb['G']) if GhAb['G'] is not None else None
h = np.asarray(GhAb['h']) if GhAb['h'] is not None else None
A = np.asarray(GhAb['A']) if GhAb['A'] is not None else None
b = np.asarray(GhAb['b']) if GhAb['b'] is not None else None
G = G if (G is not None and G.size > 0) else None
h = h if (h is not None and h.size > 0) else None
A = A if (A is not None and A.size > 0) else None
b = b if (b is not None and b.size > 0) else None

def primal_residual(w, G, h, A, b):
    res = 0.0
    
    if G is not None:
        res = max(res, np.max(G @ w - h))
        
    if A is not None:
        res = max(res, np.max(np.abs(A @ w - b)))
        
    return res


def dual_residual(P_qp, q_qp, w):
    # Approximation: ||Pw - q||
    return np.linalg.norm(P_qp @ w - q_qp)


def duality_gap(P_qp, q_qp, w):
    # Approximation (since dual vars not always available)
    return abs(w @ (P_qp @ w) / 2 - q_qp @ w)


# Loop over solvers, instantiate the quadratic program, solve it and store the results in a dictionary.
solvers = USABLE_SOLVERS

results_dict = {}

for solver in solvers:
    try:
        start = time.time()
        
        w = solve_qp(
            P_qp,
            q_qp,
            G=G,
            h=h,
            A=A,
            b=b,
            solver=solver
        )
        
        runtime = time.time() - start
        
        if w is None:
            raise ValueError("Solver returned None")
        
        obj = objective(P_qp, q_qp, w)  # careful: original q
        
        results_dict[solver] = {
            'solution_found': True,
            'objective': obj,
            'primal_residual': primal_residual(w, G, h, A, b),
            'dual_residual': dual_residual(P_qp, q_qp, w),
            'duality_gap': duality_gap(P_qp, q_qp, w),
            'runtime': runtime,
        }
        
    except Exception as e:
        #print(f"Solver {solver} failed with error: {e}")
        results_dict[solver] = {
            'solution_found': False,
            'objective': np.nan,
            'primal_residual': np.nan,
            'dual_residual': np.nan,
            'duality_gap': np.nan,
            'runtime': np.nan,
        }

results = pd.DataFrame(results_dict)

avaliable solvers: ['clarabel', 'daqp', 'highs', 'osqp', 'qpalm', 'quadprog', 'scs']
P shape: (50, 50)
q shape: (50,)


/Users/danfriedman/Documents/Quantitative Portfolio Management w Python/Quantitative-Portfolio-Management-w-Python/.venv/lib/python3.12/site-packages/qpsolvers/conversions/ensure_sparse_matrices.py:28: SparseConversionWarning: Converted matrix 'P' of your problem to scipy.sparse.csc_matrix to pass it to solver 'qpalm'; for best performance, build your matrix as a csc_matrix directly.
  warnings.warn(
/Users/danfriedman/Documents/Quantitative Portfolio Management w Python/Quantitative-Portfolio-Management-w-Python/.venv/lib/python3.12/site-packages/qpsolvers/conversions/ensure_sparse_matrices.py:28: SparseConversionWarning: Converted matrix 'G' of your problem to scipy.sparse.csc_matrix to pass it to solver 'qpalm'; for best performance, build your matrix as a csc_matrix directly.
  warnings.warn(
/Users/danfriedman/Documents/Quantitative Portfolio Management w Python/Quantitative-Portfolio-Management-w-Python/.venv/lib/python3.12/site-packages/qpsolvers/conversions/ensure_sparse_matric

Print and visualize the results

In [18]:
print(results)

                cvxopt   daqp     qpalm      osqp  quadprog
solution_found   False  False      True      True      True
objective          NaN    NaN   0.00003  0.000001  0.000157
primal_residual    NaN    NaN  0.000001  0.000283  3.105243
dual_residual      NaN    NaN  0.000162  0.000162  0.000162
duality_gap        NaN    NaN   0.00003  0.000001  0.000157
runtime            NaN    NaN  0.007793  0.002617  0.000695


## 2. Analytical Solution to Minimum-Variance Problem

(5 points)

- Create a `MinVariance` class that follows the structure of the `MeanVariance` class.
- Implement the `solve` method in `MinVariance` such that if `solver_name = 'analytical'`, the analytical solution is computed and stored within the object (if such a solution exists). If not, call the `solve` method from the parent class.
- Create a `Constraints` object by injecting the same ids as in part 1.b) and add a budget constraint.
- Instantiate a `MinVariance` object by setting `solver_name = 'analytical'` and passing instances of `Constraints` and `Covariance` as arguments.
- Create an `OptimizationData` object that contains an element `return_series`, which consists of the synthetic data generated in part 1.a).
- Solve the optimization problem using the created `MinVariance` object and compare the results to those obtained in part 1.c).


In [6]:
# Define class MinVariance
class MinVariance(Optimization):

    def __init__(self,
                 constraints: Constraints,
                 covariance: Optional[Covariance] = None,
                 **kwargs):
        super().__init__(
            constraints=constraints,
            **kwargs
        )
        self.covariance = Covariance() if covariance is None else covariance

    def set_objective(self, optimization_data: OptimizationData) -> None:
        # <your code here>

    def solve(self) -> None:
        if self.params.get('solver_name') == 'analytical':
            # <your code here>
            return None
        else:
            return super().solve()


# Create a constraints object with just a budget constraint
# <your code here>

# Instantiate the MinVariance class
# <your code here>

# Prepare the optimization data and prepare the optimization problem
# <your code here>

# Solve the optimization problem and print the weights
# <your code here>

IndentationError: expected an indented block after function definition on line 14 (3954588042.py, line 17)